# Tahap 5: Integrasi Sistem Rekomendasi Hibrida ARM-LSTM dan Validasi Efektivitas Line-Up
**Penelitian:** HOK Virtual Coach: Model Rekomendasi Draft Pick pada Game Honor of Kings Berbasis Association Rule Mining (ARM) dan Long Short-Term Memory (LSTM)  
**Oleh:** Adiyatma Admaja - NIM 082111633061 (S-1 Sistem Informasi, Universitas Airlangga)  
**Referensi Naskah:** BAB II Subbab 2.5, BAB III Subbab 3.4 & 3.5, serta BAB IV (Hasil Integrasi Sistem dan Validasi Simulasi)  
**Urutan Pipeline:** Tahap 5 dari 5 (Tahap akhir; mengintegrasikan seluruh artefak dari Notebook 1, 2, 3, dan 4)

## 1. Tujuan Notebook
Notebook ini bertujuan mengintegrasikan pengetahuan aturan asosiasi (ARM) dan prediksi sekuensial (LSTM) ke dalam arsitektur rekomendasi hibrida dua tingkat (*Two-Tier Hybrid*), serta memvalidasi efektivitas komposisi *line-up* yang dihasilkan melalui simulasi *draft pick* bergantian (*turn-based*) menggunakan model evaluator Multi-Layer Perceptron (MLP).

## 2. Spesifikasi Input
* **Parameter & Kamus Hero (Output Tahap 1):** `Data/global_parameters.pkl` dan `Data/hero_data.csv` (mencakup atribut `Default_Role`, `Total_Picks`, dan `Win_Rate`).
* **Basis Aturan ARM (Output Tahap 2):** Kelima berkas CSV pada folder `Extracted-ARM-Rules/` (`rules_allies.csv`, `rules_counters.csv`, `rules_meta_ban.csv`, `rules_combo_breaker.csv`, dan `rules_anti_counter.csv`).
* **Model Deep Learning Terlatih (Output Tahap 3 & 4):** `Model/lstm_draft_pick_model.keras` dan `Model/mlp_match_prediction_model.keras`.

## 3. Alur Mekanisme Integrasi & Skenario Eksperimen
1. **Role Constraint Filter (Subbab 3.4.1):** Menyaring daftar *hero* yang masih tersedia agar tidak terjadi duplikasi peran (*Clash Lane, Jungler, Mid Lane, Farm Lane, Roamer*) di dalam formasi tim pada fase *Pick* (dinonaktifkan pada fase *Ban*).
2. **Ekstraksi Subset & Normalisasi Skor (Subbab 3.4.2):**
   * *Subset LSTM:* Mengambil 18 kandidat *hero* belum terpilih dengan probabilitas Softmax tertinggi.
   * *Subset ARM:* Mengevaluasi aturan asosiasi secara kontekstual sesuai fase (*Ban Phase 1*: Meta Ban; *Pick Phase 1 & 2*: Allies & Counters; *Ban Phase 2*: Anti-Counter & Combo Breaker), lalu menormalisasi akumulasi skornya ke rentang `[0, 1]` menggunakan *Max Normalization* (Persamaan 3.3).
3. **Keputusan Two-Tier Hybrid & Dynamic Weighting (Tabel 3.8):** Menghitung skor fusi $skorAkhir_i = (w_{arm} \times skorARM_i) + (w_{lstm} \times probabilitasLSTM_i)$ dengan bobot dinamis: **Ban 1** $(0.7, 0.3)$, **Pick 1** $(0.6, 0.4)$, **Ban 2** $(0.4, 0.6)$, dan **Pick 2** $(0.3, 0.7)$. Sistem memprioritaskan kandidat pada **Tier 1 (Irisan Subset)** antara kandidat ARM valid dan Top-18 LSTM, serta mengaktifkan **Tier 2 (Fallback Weighted Average)** apabila tidak ditemukan irisan.
4. **Simulasi Draft Pick & Evaluasi MLP (Subbab 3.5):** Menjalankan simulasi 18 tindakan *draft* (`b1-b2-b1-b2-p1-p2-p2-p1-p1-p2-b2-b1-b2-b1-p2-p1-p1-p2`) sebanyak **100 iterasi per skenario** yang mempertemukan **Tim Biru** (`HYBRID`, `ARM_ONLY`, `LSTM_ONLY` pada tingkat pemilihan `Top-1`, `Top-3`, dan `Top-5`) melawan **Tim Merah** (*Random Draft* [`RD`] dan *Highest Win Rate* [`HWR`]). Komposisi *line-up* lengkap yang terbentuk dikonversi ke matriks *Multi-Hot Encoding* dan dievaluasi oleh model MLP (`prob_menang > 0.5` dikategorikan sebagai kemenangan Tim Biru).

## 4. Artefak Output yang Dihasilkan
* Bedah tahapan proses rekomendasi *Two-Tier Hybrid* pada langkah *draft* spesifik.
* Tabel Pivot Komparasi **Win Rate (%)** Tim Biru dari 100 pertandingan simulasi di seluruh 18 skenario pengujian beserta pemeriksaan keunikan komposisi *line-up*.

In [1]:
import pandas as pd
import numpy as np
import random
import pickle
import time
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# ==============================================================================
# 1. LOAD PARAMETER GLOBAL & HERO LIST
# ==============================================================================
with open('Data/global_parameters.pkl', 'rb') as f:
    global_params = pickle.load(f)

encoder = global_params['encoder']
MAX_INPUT_LENGTH = global_params['MAX_INPUT_LENGTH']
TOTAL_HEROES_VOCAB = global_params['TOTAL_HEROES_VOCAB']

# Dataset Hero untuk referensi nama dan ID
df_hero = pd.read_csv("Data/hero_data.csv")

# ------------------------------------------------------------------------------
# PRE-COMPUTED DICTIONARY UNTUK LOOKUP CEPAT (O(1))
# ------------------------------------------------------------------------------
hero_name_to_idx = {}
for _, row in df_hero.iterrows():
    hero_name_to_idx[row['Hero_Name']] = encoder.transform([row['Hero_ID']])[0] + 1

idx_to_hero_name = {v: k for k, v in hero_name_to_idx.items()}

# Dictionary untuk role filtering
dict_role_hero = dict(zip(df_hero['Hero_Name'], df_hero['Default_Role']))

In [3]:
# ==============================================================================
# 2. LOAD MODEL DEEP LEARNING (LSTM & MLP)
# ==============================================================================
model_lstm = load_model('Model/lstm_draft_pick_model.keras')
model_mlp = load_model('Model/mlp_match_prediction_model.keras')

In [4]:
# ==============================================================================
# 3. LOAD & PRE-COMPUTE DATA ATURAN ASOSIASI (ARM RULE BASES)
# ==============================================================================
parsed_rules_allies = []
parsed_rules_counters = []
parsed_rules_meta_ban = []
parsed_rules_combo_breaker = []
parsed_rules_anti_counter = []

try:
    # Load raw CSV
    rules_allies = pd.read_csv('Extracted-ARM-Rules/rules_allies.csv')
    rules_counters = pd.read_csv('Extracted-ARM-Rules/rules_counters.csv')
    rules_meta_ban = pd.read_csv('Extracted-ARM-Rules/rules_meta_ban.csv')
    rules_combo_breaker = pd.read_csv('Extracted-ARM-Rules/rules_combo_breaker.csv')
    rules_anti_counter = pd.read_csv('Extracted-ARM-Rules/rules_anti_counter.csv')

    # Pre-compute parsing ke bentuk Native Python agar lookup super cepat
    for _, row in rules_allies.iterrows():
        parsed_rules_allies.append({
            'ants': set(str(row['Team_Heroes']).split(', ')),
            'cons': str(row['Rec_Ally']).split(', ')[0],
            'score': row['win_rate']
        })

    for _, row in rules_counters.iterrows():
        parsed_rules_counters.append({
            'enemy': str(row['Enemy']),
            'cons': str(row['Rec_Counter']),
            'score': row['counter_coef']
        })

    for _, row in rules_meta_ban.iterrows():
        parsed_rules_meta_ban.append({
            'ants': set(str(row['antecedents']).split(', ')),
            'cons': str(row['Target_Ban']).split(', ')[0],
            'score': row['confidence']
        })

    for _, row in rules_combo_breaker.iterrows():
        parsed_rules_combo_breaker.append({
            'ants': set([str(i).replace('EnemyPick_', '') for i in str(row['antecedents']).split(', ')]),
            'cons': [str(i).replace('OurBan_', '') for i in str(row['consequents']).split(', ')][0],
            'score': row['confidence']
        })

    for _, row in rules_anti_counter.iterrows():
        parsed_rules_anti_counter.append({
            'ants': set([str(i).replace('OurPick_', '') for i in str(row['antecedents']).split(', ')]),
            'cons': [str(i).replace('OurBan_', '') for i in str(row['consequents']).split(', ')][0],
            'score': row['confidence']
        })

    print("Semua data aturan ARM berhasil dimuat dan di-parsing untuk Fast Lookup!")
except FileNotFoundError:
    print("File aturan ARM tidak ditemukan.")

Semua data aturan ARM berhasil dimuat dan di-parsing untuk Fast Lookup!


In [5]:
# ==============================================================================
# 4. FUNGSI PENYARING PERAN (ROLE CONSTRAINT FILTER)
# ==============================================================================
def dapatkan_kandidat_valid_role(daftar_hero_tersedia, tim_saat_ini, dict_role):
    """
    Fungsi ini menyaring hero yang rolenya tumpang tindih.
    Sesuai instruksi Dosbing: Jika role belum lengkap, harus seleksi hero role baru.
    """
    # 1. Identifikasi role apa saja yang sudah terisi di tim saat ini
    role_terisi = set()
    for hero in tim_saat_ini:
        if hero in dict_role:
            role_terisi.add(dict_role[hero])

    # 2. Saring daftar_hero_tersedia, hanya simpan yang rolenya BELUM ada di tim
    kandidat_valid = []
    for hero in daftar_hero_tersedia:
        role_kandidat = dict_role.get(hero)
        if role_kandidat not in role_terisi:
            kandidat_valid.append(hero)

    # 3. Jika semua 5 role utama sudah terisi (atau tidak ada hero sisa dengan role baru),
    # barulah sistem bebas memilih dari semua hero yang masih tersedia (komposisi lengkap).
    if not kandidat_valid:
        return daftar_hero_tersedia

    return kandidat_valid

In [6]:
# ==============================================================================
# 5. FUNGSI REKOMENDASI HIBRIDA DINAMIS (ARM + LSTM)
# ==============================================================================
def rekomendasi_dinamis_mode(urutan_draft_sekarang, our_picks, enemy_picks,
                             model_lstm, parsed_rules_allies, parsed_rules_counters,
                             parsed_rules_meta_ban, parsed_rules_combo_breaker, parsed_rules_anti_counter,
                             MAX_INPUT_LENGTH, df_hero, dict_role, mode='hybrid', top_k=3):

    draft_len = len(urutan_draft_sekarang)
    is_ban_phase_1  = draft_len < 4
    is_pick_phase_1 = 4 <= draft_len < 10
    is_ban_phase_2  = 10 <= draft_len < 14
    is_pick_phase_2 = 14 <= draft_len < 18
    is_ban_phase  = is_ban_phase_1 or is_ban_phase_2

    # Distribusi Bobot (Hanya digunakan untuk Fallback / Mode Tunggal)
    if mode == 'arm_only': w_arm, w_lstm = 1.0, 0.0
    elif mode == 'lstm_only': w_arm, w_lstm = 0.0, 1.0
    else:
        if is_ban_phase_1: w_arm, w_lstm = 0.7, 0.3
        elif is_pick_phase_1: w_arm, w_lstm = 0.6, 0.4
        elif is_ban_phase_2: w_arm, w_lstm = 0.4, 0.6
        else: w_arm, w_lstm = 0.3, 0.7

    # --- 1. KUMPULKAN KANDIDAT VALID (STRICT ROLE FILTER) ---
    semua_hero = set(df_hero['Hero_Name'])
    hero_tersedia = list(semua_hero - set(urutan_draft_sekarang))

    if not is_ban_phase:
        kandidat_valid_role = dapatkan_kandidat_valid_role(hero_tersedia, our_picks, dict_role)
    else:
        kandidat_valid_role = hero_tersedia # Fase Ban tidak butuh role filter

    # --- 2. EKSEKUSI LSTM ---
    encoded_input = [hero_name_to_idx.get(nama, 0) for nama in urutan_draft_sekarang]
    padded_input = pad_sequences([encoded_input], maxlen=MAX_INPUT_LENGTH, padding='pre')
    lstm_probs = model_lstm.predict(padded_input, verbose=0)[0]

    # Ambil Top 18 LSTM dari hero yang belum dipilih
    for idx in encoded_input:
        lstm_probs[idx] = 0.0 # Masking hero yang sudah di-pick/ban
    top_18_idx = np.argsort(lstm_probs)[-18:][::-1]
    top_18_lstm_heroes = {idx_to_hero_name[idx] for idx in top_18_idx if idx != 0 and idx_to_hero_name[idx] in hero_tersedia}

    # --- 3. EKSEKUSI ARM ---
    arm_scores = np.zeros(TOTAL_HEROES_VOCAB)
    set_draft_sekarang = set(urutan_draft_sekarang)
    set_our_picks = set(our_picks)
    set_enemy_picks = set(enemy_picks)

    if is_ban_phase_1:
        for rule in parsed_rules_meta_ban:
            if rule['ants'].issubset(set_draft_sekarang):
                try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
                except KeyError: pass

    elif is_ban_phase_2:
        for rule in parsed_rules_anti_counter:
            if rule['ants'].issubset(set_our_picks):
                try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
                except KeyError: pass

        for rule in parsed_rules_combo_breaker:
            if rule['ants'].issubset(set_enemy_picks):
                try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
                except KeyError: pass

    elif is_pick_phase_1 or is_pick_phase_2:
        for rule in parsed_rules_allies:
            if rule['ants'].issubset(set_our_picks):
                try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
                except KeyError: pass

        for rule in parsed_rules_counters:
            if rule['enemy'] in set_enemy_picks:
                try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
                except KeyError: pass

    # Normalisasi Skor ARM (Min-Max)
    max_arm = np.max(arm_scores)
    if max_arm > 0: arm_scores = arm_scores / max_arm

    # --- 4. KEPUTUSAN FINAL: TIER 1 (IRISAN) ATAU TIER 2 (FALLBACK) ---
    skor_akhir = (w_arm * arm_scores) + (w_lstm * lstm_probs)
    rekomendasi = []

    if mode == 'hybrid':
        # Ambil hero yang lolos role filter & punya skor aturan ARM > 0
        kandidat_arm_valid = [h for h in kandidat_valid_role if arm_scores[hero_name_to_idx.get(h, 0)] > 0]

        # TIER 1: Cek Irisan/Subset dengan Top 18 LSTM
        irisan = [h for h in kandidat_arm_valid if h in top_18_lstm_heroes]

        if irisan:
            # Jika ada irisan, urutkan berdasarkan skor probabilitas tertinggi
            irisan.sort(key=lambda h: skor_akhir[hero_name_to_idx.get(h, 0)], reverse=True)
            rekomendasi = [{'Hero': h, 'Tier': '1-Subset'} for h in irisan[:top_k]]
        else:
            # TIER 2: Fallback Weighted Average pada kandidat yang VALID SECARA ROLE
            kandidat_valid_role.sort(key=lambda h: skor_akhir[hero_name_to_idx.get(h, 0)], reverse=True)
            rekomendasi = [{'Hero': h, 'Tier': '2-Fallback'} for h in kandidat_valid_role[:top_k] if skor_akhir[hero_name_to_idx.get(h, 0)] > 0]

    elif mode == 'arm_only':
        kandidat_valid_role.sort(key=lambda h: arm_scores[hero_name_to_idx.get(h, 0)], reverse=True)
        rekomendasi = [{'Hero': h, 'Tier': 'ARM'} for h in kandidat_valid_role[:top_k] if arm_scores[hero_name_to_idx.get(h, 0)] > 0]

    elif mode == 'lstm_only':
        kandidat_valid_role.sort(key=lambda h: lstm_probs[hero_name_to_idx.get(h, 0)], reverse=True)
        rekomendasi = [{'Hero': h, 'Tier': 'LSTM'} for h in kandidat_valid_role[:top_k] if lstm_probs[hero_name_to_idx.get(h, 0)] > 0]

    return rekomendasi

In [7]:
# ==============================================================================
# 6. FUNGSI STRATEGI BASELINE & WRAPPER PEMILIH TINDAKAN
# ==============================================================================
def dapatkan_aksi_agen(mode_agen, df_hero, urutan_draft, our_picks, enemy_picks, dict_role, top_k_agen=3):

    # 1. Tentukan kandidat yang valid secara Role Filter
    semua_hero = set(df_hero['Hero_Name'])
    hero_tersedia = list(semua_hero - set(urutan_draft))

    draft_len = len(urutan_draft)
    is_ban_phase = draft_len < 4 or (10 <= draft_len < 14)

    if not is_ban_phase:
        kandidat_valid = dapatkan_kandidat_valid_role(hero_tersedia, our_picks, dict_role)
    else:
        kandidat_valid = hero_tersedia # Aman saat Ban Phase

    # 2. Dapatkan rekomendasi berdasarkan mode
    if mode_agen == 'rd':
        kandidat_terpilih = random.sample(kandidat_valid, min(top_k_agen, len(kandidat_valid)))

    elif mode_agen == 'hwr':
        df_available = df_hero[df_hero['Hero_Name'].isin(kandidat_valid)]
        kandidat_terpilih = df_available.sort_values(
            by=['Total_Picks', 'Win_Rate'], ascending=[False, False]
        ).head(top_k_agen)['Hero_Name'].tolist()

    else:
        # PERBAIKAN DI SINI: Menggunakan variabel parsed_rules_...
        rek = rekomendasi_dinamis_mode(urutan_draft, our_picks, enemy_picks,
                                       model_lstm, parsed_rules_allies, parsed_rules_counters,
                                       parsed_rules_meta_ban, parsed_rules_combo_breaker, parsed_rules_anti_counter,
                                       MAX_INPUT_LENGTH, df_hero, dict_role, mode=mode_agen, top_k=top_k_agen)
        if rek:
            kandidat_terpilih = [item['Hero'] for item in rek]
        else:
            df_available = df_hero[df_hero['Hero_Name'].isin(kandidat_valid)]
            kandidat_terpilih = df_available.sort_values(
                by=['Total_Picks', 'Win_Rate'], ascending=[False, False]
            ).head(top_k_agen)['Hero_Name'].tolist()

    # 3. Eksekusi tindakan
    if kandidat_terpilih:
        return random.choice(kandidat_terpilih)
    else:
        return random.choice(kandidat_valid)

print("Seluruh Engine Agen dan Hibrida Dua Lapis berhasil diinisialisasi.")

Seluruh Engine Agen dan Hibrida Dua Lapis berhasil diinisialisasi.


In [8]:
# ==============================================================================
# 7. MESIN SIMULASI DRAFT PICK MODULAR
# ==============================================================================
pola_draft = ['b1', 'b2', 'b1', 'b2', 'p1', 'p2', 'p2', 'p1', 'p1', 'p2',
              'b2', 'b1', 'b2', 'b1', 'p2', 'p1', 'p1', 'p2']

def jalankan_simulasi_agen(mode_blue, mode_red, top_k_blue=3, top_k_red=3, jumlah_iterasi=100):
    hasil_simulasi = []

    for iterasi in range(1, jumlah_iterasi + 1):
        urutan_draft = []
        blue_picks = []
        red_picks = []

        for aksi in pola_draft:
            tim = aksi[1]
            jenis = aksi[0]

            if tim == '1': # Giliran Blue
                pilihan = dapatkan_aksi_agen(mode_blue, df_hero, urutan_draft, blue_picks, red_picks, dict_role_hero, top_k_agen=top_k_blue)
                urutan_draft.append(pilihan)
                if jenis == 'p': blue_picks.append(pilihan)
            else: # Giliran Red
                pilihan = dapatkan_aksi_agen(mode_red, df_hero, urutan_draft, red_picks, blue_picks, dict_role_hero, top_k_agen=top_k_red)
                urutan_draft.append(pilihan)
                if jenis == 'p': red_picks.append(pilihan)

        hasil_simulasi.append({
            'Skenario': f"Blue({mode_blue.upper()}) vs Red({mode_red.upper()})",
            'Blue_Lineup': blue_picks,
            'Red_Lineup': red_picks
        })

    return pd.DataFrame(hasil_simulasi)

In [9]:
# ==============================================================================
# 7. TRANSFORMASI MULTI-HOT ENCODING (BATCH) UNTUK EVALUASI PREDIKSI MLP
# ==============================================================================
def konversi_ke_multihot_batch(df_simulasi, vocab_size):
    X_eval = np.zeros((len(df_simulasi), vocab_size))
    for i, row in df_simulasi.iterrows():
        for hero in row['Blue_Lineup']:
            if hero in hero_name_to_idx:
                X_eval[i, hero_name_to_idx[hero]] = 1
        for hero in row['Red_Lineup']:
            if hero in hero_name_to_idx:
                X_eval[i, hero_name_to_idx[hero]] = -1
    return X_eval

In [10]:
# ==============================================================================
# 10. EKSEKUSI SKENARIO SIMULASI DAN EVALUASI WIN RATE
# ==============================================================================

# Inisialisasi parameter eksperimen
jumlah_iterasi_uji = 100
top_k_lawan = 10   # Variansi tim Red

# Variasi Top-K untuk tim Blue
daftar_top_k_blue = [1, 3, 5]

# Daftar Skenario Duel (Mode Blue vs Mode Red)
skenario_eksperimen = [
    ('hybrid', 'rd'),
    ('hybrid', 'hwr'),
    ('arm_only', 'rd'),
    ('arm_only', 'hwr'),
    ('lstm_only', 'rd'),
    ('lstm_only', 'hwr')
]

tabel_kemenangan = []
kumpulan_df_simulasi = {}

start_eval = time.time()
print("Memulai Eksekusi Eksperimen simulasi \n")

for top_k_val in daftar_top_k_blue:
    for mode_blue, mode_red in skenario_eksperimen:
        nama_skenario = f"Blue ({mode_blue.upper()}) vs Red ({mode_red.upper()})"
        print(f"Menjalankan: {nama_skenario} [Gunakan Blue Top-{top_k_val}]")

        # 1. Jalankan simulasi
        df_sim = jalankan_simulasi_agen(
            mode_blue=mode_blue,
            mode_red=mode_red,
            top_k_blue=top_k_val,
            top_k_red=top_k_lawan,
            jumlah_iterasi=jumlah_iterasi_uji
        )

        kunci_dict = f"{nama_skenario} [Top-{top_k_val}]"
        kumpulan_df_simulasi[kunci_dict] = df_sim

        # 2. Evaluasi komposisi tim (MLP)
        X_eval = konversi_ke_multihot_batch(df_sim, TOTAL_HEROES_VOCAB)
        prob_menang = model_mlp.predict(X_eval, verbose=0).flatten()

        # 3. Hitung Win Rate
        jumlah_menang_blue = np.sum(prob_menang > 0.5)
        jumlah_menang_red = jumlah_iterasi_uji - jumlah_menang_blue
        win_rate = (jumlah_menang_blue / jumlah_iterasi_uji) * 100

        # 4. Simpan ke tabel
        tabel_kemenangan.append({
            'Skenario': nama_skenario,
            'Top_K_Blue': f"Top {top_k_val}",
            'Blue_Win': jumlah_menang_blue,
            'Red_Win': jumlah_menang_red,
            'Win_Rate': win_rate
        })

print(f"\nEksperimen selesai dalam {time.time() - start_eval:.2f} detik.")

Memulai Eksekusi Eksperimen simulasi 

Menjalankan: Blue (HYBRID) vs Red (RD) [Gunakan Blue Top-1]
Menjalankan: Blue (HYBRID) vs Red (HWR) [Gunakan Blue Top-1]
Menjalankan: Blue (ARM_ONLY) vs Red (RD) [Gunakan Blue Top-1]
Menjalankan: Blue (ARM_ONLY) vs Red (HWR) [Gunakan Blue Top-1]
Menjalankan: Blue (LSTM_ONLY) vs Red (RD) [Gunakan Blue Top-1]
Menjalankan: Blue (LSTM_ONLY) vs Red (HWR) [Gunakan Blue Top-1]
Menjalankan: Blue (HYBRID) vs Red (RD) [Gunakan Blue Top-3]
Menjalankan: Blue (HYBRID) vs Red (HWR) [Gunakan Blue Top-3]
Menjalankan: Blue (ARM_ONLY) vs Red (RD) [Gunakan Blue Top-3]
Menjalankan: Blue (ARM_ONLY) vs Red (HWR) [Gunakan Blue Top-3]
Menjalankan: Blue (LSTM_ONLY) vs Red (RD) [Gunakan Blue Top-3]
Menjalankan: Blue (LSTM_ONLY) vs Red (HWR) [Gunakan Blue Top-3]
Menjalankan: Blue (HYBRID) vs Red (RD) [Gunakan Blue Top-5]
Menjalankan: Blue (HYBRID) vs Red (HWR) [Gunakan Blue Top-5]
Menjalankan: Blue (ARM_ONLY) vs Red (RD) [Gunakan Blue Top-5]
Menjalankan: Blue (ARM_ONLY) vs 

In [11]:
# ==============================================================================
# 9. VISUALISASI CONTOH PROSES REKOMENDASI
# ==============================================================================
def visualisasi_contoh_proses(urutan_draft_contoh, our_picks, enemy_picks, dict_role):
    print("=" * 80)
    print(f"BEDAH PROSES REKOMENDASI TWO-TIER HYBRID (CONTOH SEQUENCE KE-{len(urutan_draft_contoh)})")
    print("=" * 80)
    print(f"Draft Saat Ini : {urutan_draft_contoh}")
    print(f"Tim Kita (Our) : {our_picks}")
    print(f"Tim Lawan (Enemy): {enemy_picks}")
    print("-" * 80)

    # 1. Kumpulkan Kandidat Valid (Role Filter)
    semua_hero = set(df_hero['Hero_Name'])
    hero_tersedia = list(semua_hero - set(urutan_draft_contoh))
    kandidat_valid_role = dapatkan_kandidat_valid_role(hero_tersedia, our_picks, dict_role)
    print(f"[TAHAP 1] Role Filter: Tersisa {len(kandidat_valid_role)} hero yang tidak bentrok role.")

    # 2. Eksekusi LSTM (Top 18 Subset)
    encoded_input = [hero_name_to_idx.get(nama, 0) for nama in urutan_draft_contoh]
    padded_input = pad_sequences([encoded_input], maxlen=MAX_INPUT_LENGTH, padding='pre')
    lstm_probs = model_lstm.predict(padded_input, verbose=0)[0]

    for idx in encoded_input:
        lstm_probs[idx] = 0.0 # Masking
    top_18_idx = np.argsort(lstm_probs)[-18:][::-1]
    top_18_lstm_heroes = {idx_to_hero_name[idx] for idx in top_18_idx if idx != 0 and idx_to_hero_name[idx] in hero_tersedia}

    print("\n[TAHAP 2] Subset LSTM (Top 18 Probabilitas Tertinggi):")
    sorted_lstm_for_display = sorted(list(top_18_lstm_heroes), key=lambda h: lstm_probs[hero_name_to_idx.get(h, 0)], reverse=True)
    for i, h in enumerate(sorted_lstm_for_display[:18]):
        print(f"  {i+1}. {h} (Prob: {lstm_probs[hero_name_to_idx.get(h, 0)]:.4f})")

    # 3. Eksekusi ARM (Subset Sinergi & Counter) dengan FAST LOOKUP
    arm_scores = np.zeros(TOTAL_HEROES_VOCAB)
    set_our_picks = set(our_picks)
    set_enemy_picks = set(enemy_picks)

    for rule in parsed_rules_allies:
        if rule['ants'].issubset(set_our_picks):
            try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
            except KeyError: pass

    for rule in parsed_rules_counters:
        if rule['enemy'] in set_enemy_picks:
            try: arm_scores[hero_name_to_idx[rule['cons']]] += rule['score']
            except KeyError: pass

    max_arm = np.max(arm_scores)
    if max_arm > 0: arm_scores = arm_scores / max_arm

    kandidat_arm_valid = [h for h in kandidat_valid_role if arm_scores[hero_name_to_idx.get(h, 0)] > 0]
    kandidat_arm_valid.sort(key=lambda h: arm_scores[hero_name_to_idx.get(h, 0)], reverse=True)

    print(f"\n[TAHAP 3] Subset ARM (Kombinasi Sinergi/Counter yang Valid Role): Ditemukan {len(kandidat_arm_valid)} hero.")
    for i, h in enumerate(kandidat_arm_valid[:18]):
        print(f"  {i+1}. {h} (Skor ARM: {arm_scores[hero_name_to_idx.get(h, 0)]:.4f})")

    # ---------------------------------------------------------
    # PEMBOBOTAN DINAMIS (Berdasarkan panjang draft saat ini)
    # ---------------------------------------------------------
    draft_len = len(urutan_draft_contoh)
    if draft_len < 4:
        w_arm, w_lstm, nama_fase = 0.7, 0.3, "Ban Phase 1"
    elif draft_len < 10:
        w_arm, w_lstm, nama_fase = 0.6, 0.4, "Pick Phase 1"
    elif draft_len < 14:
        w_arm, w_lstm, nama_fase = 0.4, 0.6, "Ban Phase 2"
    else:
        w_arm, w_lstm, nama_fase = 0.3, 0.7, "Pick Phase 2"

    skor_akhir = (w_arm * arm_scores) + (w_lstm * lstm_probs)
    irisan = [h for h in kandidat_arm_valid if h in top_18_lstm_heroes]

    print(f"\n[TAHAP 4] KEPUTUSAN HYBRID (Fase: {nama_fase} | Bobot -> ARM: {w_arm}, LSTM: {w_lstm}):")
    if irisan:
        irisan.sort(key=lambda h: skor_akhir[hero_name_to_idx.get(h, 0)], reverse=True)
        print(f"  >>> STATUS: IRISAN DITEMUKAN (Tier 1)")
        print(f"  >>> Rekomendasi Teratas: 1. {irisan[0]} (Tertebak LSTM & Punya aturan ARM)")

        # Tambahan Rekomendasi Peringkat 2 hingga 5
        if len(irisan) > 1:
            print(f"  >>> Rekomendasi Tambahan:")
            for i, h in enumerate(irisan[1:5], start=2):
                print(f"      {i}. {h}")

    else:
        kandidat_valid_role.sort(key=lambda h: skor_akhir[hero_name_to_idx.get(h, 0)], reverse=True)
        print(f"  >>> STATUS: IRISAN KOSONG! Menggunakan Fallback Weighted Average (Tier 2)")
        print(f"  >>> Rekomendasi Teratas: 1. {kandidat_valid_role[0]}")

        # Tambahan Rekomendasi Peringkat 2 hingga 5
        if len(kandidat_valid_role) > 1:
            print(f"  >>> Rekomendasi Tambahan:")
            for i, h in enumerate(kandidat_valid_role[1:5], start=2):
                if skor_akhir[hero_name_to_idx.get(h, 0)] > 0: # Hanya tampilkan yang punya skor
                    print(f"      {i}. {h}")
    print("=" * 80 + "\n")

# Menjalankan Contoh Demo
contoh_draft = ['Haya', 'Pei', 'Liang', 'Ao_Yin', 'Dyadia', 'Arli', 'Zhang_Fei', 'Guan_Yu', 'Augran', 'Dharma', 'Consort_Yu', 'Kaizer', 'Mai_Shiranui', 'Lorion', 'Nuwa']
contoh_our = ['Dyadia', 'Guan_Yu', 'Augran']
contoh_enemy = ['Arli', 'Zhang_Fei', 'Dharma', 'Nuwa']

visualisasi_contoh_proses(contoh_draft, contoh_our, contoh_enemy, dict_role_hero)

BEDAH PROSES REKOMENDASI TWO-TIER HYBRID (CONTOH SEQUENCE KE-15)
Draft Saat Ini : ['Haya', 'Pei', 'Liang', 'Ao_Yin', 'Dyadia', 'Arli', 'Zhang_Fei', 'Guan_Yu', 'Augran', 'Dharma', 'Consort_Yu', 'Kaizer', 'Mai_Shiranui', 'Lorion', 'Nuwa']
Tim Kita (Our) : ['Dyadia', 'Guan_Yu', 'Augran']
Tim Lawan (Enemy): ['Arli', 'Zhang_Fei', 'Dharma', 'Nuwa']
--------------------------------------------------------------------------------
[TAHAP 1] Role Filter: Tersisa 36 hero yang tidak bentrok role.

[TAHAP 2] Subset LSTM (Top 18 Probabilitas Tertinggi):
  1. Lady_Zhen (Prob: 0.0948)
  2. Luna (Prob: 0.0705)
  3. Mayene (Prob: 0.0630)
  4. Di_Renjie (Prob: 0.0595)
  5. Chano (Prob: 0.0540)
  6. Yixing (Prob: 0.0396)
  7. Musashi (Prob: 0.0371)
  8. Yuhuan (Prob: 0.0284)
  9. Agudo (Prob: 0.0240)
  10. Lian_Po (Prob: 0.0222)
  11. Biron (Prob: 0.0210)
  12. Charlotte (Prob: 0.0205)
  13. Mozi (Prob: 0.0188)
  14. Feyd (Prob: 0.0181)
  15. Shi (Prob: 0.0172)
  16. Bai_Qi (Prob: 0.0172)
  17. Wang_Zhaoj

In [12]:
# ------------------------------------------------------------------------------
# 9. TAMPILKAN HASIL EVALUASI DALAM BENTUK PIVOT TABLE
# ------------------------------------------------------------------------------
df_hasil_mentah = pd.DataFrame(tabel_kemenangan)

# Menggabungkan persentase dan detail menang-kalah
df_hasil_mentah['Tampilan_Pivot'] = df_hasil_mentah.apply(
    lambda row: f"{row['Win_Rate']:.1f}% (B:{row['Blue_Win']} | R:{row['Red_Win']})", axis=1
)

# Memisahkan nama skenario menjadi Blue dan Red
df_hasil_mentah[['Blue', 'Red']] = df_hasil_mentah['Skenario'].str.extract(r'Blue \((.*?)\) vs Red \((.*?)\)')

# Mapping nama skenario
nama_blue = {
    'ARM_ONLY': 'ARM',
    'LSTM_ONLY': 'LSTM',
    'HYBRID': 'Hybrid'
}

nama_red = {
    'HWR': 'Highest Win Rate',
    'RD': 'Random Draft'
}

# Mengganti nama skenario
df_hasil_mentah['Blue'] = df_hasil_mentah['Blue'].replace(nama_blue)
df_hasil_mentah['Red'] = df_hasil_mentah['Red'].replace(nama_red)

# Urutan Kustom untuk kolom Blue
urutan_blue = ['ARM', 'LSTM', 'Hybrid']
df_hasil_mentah['Blue'] = pd.Categorical(
    df_hasil_mentah['Blue'],
    categories=urutan_blue,
    ordered=True)

# Membuat pivot table dasar
df_pivot = df_hasil_mentah.pivot(
    index=['Blue', 'Red'],
    columns='Top_K_Blue',
    values='Tampilan_Pivot'
).reset_index()

# Mengurutkan kolom
kolom_urutan = ['Blue', 'Red', 'Top 1', 'Top 3', 'Top 5']
df_pivot = df_pivot.reindex(columns=kolom_urutan)

# Membuat Header Bertingkat (MultiIndex)
df_pivot.columns = pd.MultiIndex.from_tuples([
    ('Skenario', 'Blue'),
    ('Skenario', 'Red'),
    ('Top_K_Blue', 'Top 1'),
    ('Top_K_Blue', 'Top 3'),
    ('Top_K_Blue', 'Top 5')
])

df_pivot.insert(0, ('', 'No'), range(1, len(df_pivot) + 1))  # Menambahkan kolom nomor urut

print("\n" + "=" * 90)
print("TABEL KOMPARASI WIN RATE BLUE (HYBRID/ARM/LSTM) BERDASARKAN TOP-K")
print("Catatan: B = Blue Win, R = Red Win")
print("=" * 90)
display(
    df_pivot.style
    .hide(axis="index")
    .set_properties(**{'text-align': 'center',
                       'border': '1px solid black',
                       'padding': '7px'})
    .set_table_styles([
        {
            'selector': 'table',
            'props': [
                ('border-collapse', 'collapse'),
                ('border', '2px solid black')
            ]
        },
        {
            'selector': 'th',
            'props': [
                ('border', '1px solid black'),
                ('text-align', 'center'),
                ('padding', '7px')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('border', '1px solid black'),
                ('text-align', 'center'),
                ('padding', '7px')
            ]
        }
    ])
)


TABEL KOMPARASI WIN RATE BLUE (HYBRID/ARM/LSTM) BERDASARKAN TOP-K
Catatan: B = Blue Win, R = Red Win


In [13]:
# ==============================================================================
# 10. CEK 10 BARIS PERTAMA DARI SETIAP SKENARIO SIMULASI
# ==============================================================================
for kunci, df in kumpulan_df_simulasi.items():
    if "[Top-3]" in kunci: # Ubah ke Top-3 agar variansi lebih terlihat
        judul_bersih = kunci.replace(" [Top-3]", "").replace("Blue (", "").replace(") vs Red (", " vs ").replace(")", "")
        print("==========================================================================")
        print(f"Menampilkan 10 baris pertama hasil simulasi {judul_bersih}:")
        print("==========================================================================")
        display(df.head(10))
        print("\n")

Menampilkan 10 baris pertama hasil simulasi HYBRID vs RD:


,Skenario,Blue_Lineup,Red_Lineup
0,Blue(HYBRID) vs Red(RD),"[Ao_Yin, Dharma, Dolia, Mozi, Lam]","[Lian_Po, Dian_Wei, Luban_No7, Xiao_Qiao, Ming]"
1,Blue(HYBRID) vs Red(RD),"[Menki, Dharma, Zhang_Fei, Lady_Sun, Wang_Zhao...","[Dolia, Luban_No7, Shangguan, Lian_Po, Feyd]"
2,Blue(HYBRID) vs Red(RD),"[Dharma, Arli, Nuwa, Musashi, Dyadia]","[Mi_Yue, Gao, Ying, Ming, Lady_Sun]"
3,Blue(HYBRID) vs Red(RD),"[Ao_Yin, Yuhuan, Agudo, Sakeer, Charlotte]","[Wang_Zhaojun, Fuzi, Gao_Changgong, Liu_Shan, ..."
4,Blue(HYBRID) vs Red(RD),"[Mai_Shiranui, Zhang_Fei, Ao_Yin, Fatih, Feyd]","[Devara, Cai_Yan, Augran, Lady_Sun, Xiao_Qiao]"
5,Blue(HYBRID) vs Red(RD),"[Mai_Shiranui, Augran, Di_Renjie, Charlotte, Z...","[Zhuangzi, Mayene, Arke, Fang, Yuhuan]"
6,Blue(HYBRID) vs Red(RD),"[Zhang_Fei, Dharma, Luna, Wang_Zhaojun, Lady_Sun]","[Han_Xin, Marco_Polo, Angela, Cai_Yan, Nezha]"
7,Blue(HYBRID) vs Red(RD),"[Ao_Yin, Zhang_Fei, Musashi, Fatih, Mozi]","[Alessio, Sima_Yi, Sakeer, Nezha, Diaochan]"
8,Blue(HYBRID) vs Red(RD),"[Pei, Mai_Shiranui, Mayene, Chano, Dyadia]","[Wang_Zhaojun, Yang_Jiang, Guiguzi, Consort_Yu..."
9,Blue(HYBRID) vs Red(RD),"[Zhang_Fei, Lady_Sun, Dharma, Yuhuan, Augran]","[Marco_Polo, Chicha, Pei, Gao, Dyadia]"




Menampilkan 10 baris pertama hasil simulasi HYBRID vs HWR:


,Skenario,Blue_Lineup,Red_Lineup
0,Blue(HYBRID) vs Red(HWR),"[Arli, Yuhuan, Dyadia, Charlotte, Feyd]","[Ao_Yin, Mayene, Augran, Wang_Zhaojun, Donghuang]"
1,Blue(HYBRID) vs Red(HWR),"[Mai_Shiranui, Menki, Zhang_Fei, Charlotte, Ch...","[Dyadia, Ao_Yin, Guan_Yu, Shi, Nakoruru]"
2,Blue(HYBRID) vs Red(HWR),"[Haya, Agudo, Yang_Jiang, Lady_Sun, Sakeer]","[Dyadia, Yuhuan, Dharma, Di_Renjie, Luna]"
3,Blue(HYBRID) vs Red(HWR),"[Dolia, Di_Renjie, Haya, Agudo, Charlotte]","[Dharma, Nuwa, Menki, Sakeer, Flowborn_Marksman]"
4,Blue(HYBRID) vs Red(HWR),"[Ao_Yin, Menki, Yuhuan, Dharma, Zhang_Fei]","[Augran, Lady_Zhen, Biron, Lady_Sun, Lapulapu]"
5,Blue(HYBRID) vs Red(HWR),"[Pei, Dyadia, Yixing, Charlotte, Flowborn_Mark...","[Di_Renjie, Biron, Augran, Liang, Cai_Yan]"
6,Blue(HYBRID) vs Red(HWR),"[Zhang_Fei, Guan_Yu, Pei, Ziya, Lady_Sun]","[Lian_Po, Menki, Dyadia, Marco_Polo, Lady_Zhen]"
7,Blue(HYBRID) vs Red(HWR),"[Arli, Wang_Zhaojun, Pei, Charlotte, Zhang_Fei]","[Lian_Po, Augran, Yuhuan, Di_Renjie, Donghuang]"
8,Blue(HYBRID) vs Red(HWR),"[Luna, Liang, Dyadia, Charlotte, Flowborn_Mark...","[Biron, Yuhuan, Chano, Zhang_Fei, Agudo]"
9,Blue(HYBRID) vs Red(HWR),"[Dyadia, Yixing, Marco_Polo, Guan_Yu, Feyd]","[Mai_Shiranui, Biron, Menki, Flowborn_Marksman..."




Menampilkan 10 baris pertama hasil simulasi ARM_ONLY vs RD:


,Skenario,Blue_Lineup,Red_Lineup
0,Blue(ARM_ONLY) vs Red(RD),"[Zhang_Fei, Pei, Flowborn_Marksman, Nuwa, Fatih]","[Charlotte, Fang, Nakoruru, Yaria, Dr_Bian]"
1,Blue(ARM_ONLY) vs Red(RD),"[Biron, Donghuang, Flowborn_Marksman, Haya, Luna]","[Musashi, Erin, Yuhuan, Charlotte, Sun_Bin]"
2,Blue(ARM_ONLY) vs Red(RD),"[Biron, Sun_Bin, Agudo, Marco_Polo, Mai_Shiranui]","[Nezha, Yaria, Shouyue, Gao_Changgong, Gao]"
3,Blue(ARM_ONLY) vs Red(RD),"[Zhang_Fei, Liang, Liu_Bang, Pei, Lady_Sun]","[Dyadia, Arli, Zhou_Yu, Butterfly, Wuyan]"
4,Blue(ARM_ONLY) vs Red(RD),"[Ao_Yin, Nezha, Pei, Yixing, Dyadia]","[Feyd, Zhuangzi, Lady_Zhen, Kaizer, Luara]"
5,Blue(ARM_ONLY) vs Red(RD),"[Chano, Dolia, Chicha, Xiao_Qiao, Pei]","[Huang_Zhong, Kongming, Musashi, Yuan_Ge, Da_Q..."
6,Blue(ARM_ONLY) vs Red(RD),"[Chano, Gan_And_Mo, Menki, Biron, Cai_Yan]","[Dyadia, Haya, Bai_Qi, Athena, Garo]"
7,Blue(ARM_ONLY) vs Red(RD),"[Dharma, Cai_Yan, Nakoruru, Liang, Arli]","[Arthur, Chano, Diaochan, Ying, Dyadia]"
8,Blue(ARM_ONLY) vs Red(RD),"[Biron, Marco_Polo, Luna, Wang_Zhaojun, Sakeer]","[Garuda, Augran, Yuan_Ge, Arli, Dolia]"
9,Blue(ARM_ONLY) vs Red(RD),"[Dharma, Shi, Dyadia, Menki, Arli]","[Biron, Athena, Daji, Shouyue, Guiguzi]"


... (2 outputs left out)



Menampilkan 10 baris pertama hasil simulasi LSTM_ONLY vs RD:


,Skenario,Blue_Lineup,Red_Lineup
0,Blue(LSTM_ONLY) vs Red(RD),"[Ao_Yin, Zhang_Fei, Lian_Po, Mozi, Musashi]","[Kaizer, Li_Bai, Lady_Zhen, Donghuang, Alessio]"
1,Blue(LSTM_ONLY) vs Red(RD),"[Dharma, Zhang_Fei, Nuwa, Chano, Lam]","[Erin, Wang_Zhaojun, Li_Bai, Arthur, Lapulapu]"
2,Blue(LSTM_ONLY) vs Red(RD),"[Zhang_Fei, Dharma, Arli, Nuwa, Feyd]","[Lapulapu, Arke, Mai_Shiranui, Umbrosa, Hou_Yi]"
3,Blue(LSTM_ONLY) vs Red(RD),"[Zhang_Fei, Chano, Yixing, Musashi, Sun_Ce]","[Mayene, Augran, Di_Renjie, Guiguzi, Haya]"
4,Blue(LSTM_ONLY) vs Red(RD),"[Ao_Yin, Dharma, Zhang_Fei, Lady_Zhen, Musashi]","[Fang, Nuwa, Menki, Li_Xin, Zhuangzi]"
5,Blue(LSTM_ONLY) vs Red(RD),"[Luna, Zhang_Fei, Ao_Yin, Guan_Yu, Lady_Zhen]","[Lapulapu, Lu_Bu, Flowborn_Marksman, Liang, Arke]"
6,Blue(LSTM_ONLY) vs Red(RD),"[Pei, Zhang_Fei, Ao_Yin, Lady_Zhen, Mayene]","[Milady, Dun, Jing, Shouyue, Da_Qiao]"
7,Blue(LSTM_ONLY) vs Red(RD),"[Zhang_Fei, Di_Renjie, Mayene, Agudo, Yixing]","[Liu_Shan, Musashi, Angela, Chicha, Luban_No7]"
8,Blue(LSTM_ONLY) vs Red(RD),"[Dharma, Pei, Lady_Zhen, Zhang_Fei, Lady_Sun]","[Dyadia, Ukyo_Tachibana, Biron, Angela, Fang]"
9,Blue(LSTM_ONLY) vs Red(RD),"[Zhang_Fei, Menki, Di_Renjie, Mayene, Liang]","[Pei, Kongming, Lady_Sun, Biron, Kui]"




Menampilkan 10 baris pertama hasil simulasi LSTM_ONLY vs HWR:


,Skenario,Blue_Lineup,Red_Lineup
0,Blue(LSTM_ONLY) vs Red(HWR),"[Ao_Yin, Lian_Po, Luna, Nuwa, Dyadia]","[Yuhuan, Zhang_Fei, Agudo, Charlotte, Fang]"
1,Blue(LSTM_ONLY) vs Red(HWR),"[Dharma, Menki, Gan_And_Mo, Sun_Bin, Lady_Sun]","[Dyadia, Ao_Yin, Augran, Guan_Yu, Yixing]"
2,Blue(LSTM_ONLY) vs Red(HWR),"[Arli, Yixing, Augran, Biron, Dyadia]","[Zhang_Fei, Pei, Lian_Po, Lady_Sun, Wang_Zhaojun]"
3,Blue(LSTM_ONLY) vs Red(HWR),"[Arli, Augran, Mayene, Wang_Zhaojun, Zhuangzi]","[Yixing, Dharma, Menki, Erin, Liu_Shan]"
4,Blue(LSTM_ONLY) vs Red(HWR),"[Mai_Shiranui, Zhang_Fei, Di_Renjie, Agudo, Gu...","[Mayene, Yixing, Musashi, Lady_Sun, Dolia]"
5,Blue(LSTM_ONLY) vs Red(HWR),"[Arli, Mai_Shiranui, Dyadia, Menki, Fatih]","[Dharma, Augran, Wang_Zhaojun, Lady_Sun, Da_Qiao]"
6,Blue(LSTM_ONLY) vs Red(HWR),"[Arli, Mai_Shiranui, Dyadia, Guan_Yu, Feyd]","[Di_Renjie, Mayene, Augran, Gan_And_Mo, Donghu..."
7,Blue(LSTM_ONLY) vs Red(HWR),"[Yuhuan, Ao_Yin, Zhang_Fei, Charlotte, Agudo]","[Arli, Lady_Zhen, Dyadia, Musashi, Fatih]"
8,Blue(LSTM_ONLY) vs Red(HWR),"[Mai_Shiranui, Dharma, Di_Renjie, Dyadia, Feyd]","[Zhang_Fei, Lady_Zhen, Musashi, Flowborn_Marks..."
9,Blue(LSTM_ONLY) vs Red(HWR),"[Luna, Dharma, Ao_Yin, Yixing, Sakeer]","[Dyadia, Di_Renjie, Agudo, Mayene, Liang]"


In [14]:
# ==============================================================================
# 11. CEK JUMLAH DUPLIKASI KOMPOSISI PADA MASING-MASING SKENARIO
# ==============================================================================
print("=" * 60)
print("JUMLAH DUPLIKAT LINE-UP (BLUE VS RED) PADA SETIAP SKENARIO")
print("=" * 60)

for kunci, df in kumpulan_df_simulasi.items():
    lineup_gabungan = df['Blue_Lineup'].astype(str) + " vs " + df['Red_Lineup'].astype(str)

    jumlah_duplikat = lineup_gabungan.duplicated().sum()
    total_data = len(df)
    jumlah_unik = total_data - jumlah_duplikat

    if "[Top-3]" in kunci:
        judul_bersih = kunci.replace(" [Top-3]", "")
        print(f"-> {judul_bersih}")
        print(f"   Unik: {jumlah_unik} match | Duplikat: {jumlah_duplikat} match (dari total {total_data})\n")

JUMLAH DUPLIKAT LINE-UP (BLUE VS RED) PADA SETIAP SKENARIO
-> Blue (HYBRID) vs Red (RD)
   Unik: 100 match | Duplikat: 0 match (dari total 100)

-> Blue (HYBRID) vs Red (HWR)
   Unik: 100 match | Duplikat: 0 match (dari total 100)

-> Blue (ARM_ONLY) vs Red (RD)
   Unik: 100 match | Duplikat: 0 match (dari total 100)

-> Blue (ARM_ONLY) vs Red (HWR)
   Unik: 100 match | Duplikat: 0 match (dari total 100)

-> Blue (LSTM_ONLY) vs Red (RD)
   Unik: 100 match | Duplikat: 0 match (dari total 100)

-> Blue (LSTM_ONLY) vs Red (HWR)
   Unik: 100 match | Duplikat: 0 match (dari total 100)

